# Collect DJ/Artist names from EDC website (2022-2025)

In [60]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [32]:
# Dictionary of EDC lineup URLs by year
edc_web_urls = {
    2022: 'https://lasvegas.electricdaisycarnival.com/past-highlights/2022-lineup/',
    2023: 'https://lasvegas.electricdaisycarnival.com/past-highlights/2023-lineup/',
    2024: 'https://lasvegas.electricdaisycarnival.com/past-highlights/2024-lineup/',
    2025: 'https://lasvegas.electricdaisycarnival.com/past-highlights/2025-lineup/'  
}

In [33]:
# Scrape artist names for all years
all_artists_data = []

for year, url in edc_web_urls.items():
    print(f"Scraping {year}...")
    
    page = requests.get(url)
    soup = BeautifulSoup(page.content, 'html')
    
    # Extract artist names from data-artist-name attributes
    artist_tags = soup.select('[data-artist-name]')
    artist_names = [tag.get('data-artist-name') for tag in artist_tags]
    
    # Add year to each artist record
    for artist in artist_names:
        all_artists_data.append({'year': year, 'artist': artist})
    
    print(f"  Found {len(artist_names)} artists for {year}")

print(f"\nTotal artists collected: {len(all_artists_data)}")

Scraping 2022...
  Found 1413 artists for 2022
Scraping 2023...
  Found 1110 artists for 2023
Scraping 2024...
  Found 1275 artists for 2024
Scraping 2025...
  Found 1287 artists for 2025

Total artists collected: 5085


In [34]:
# Create DataFrame from collected data
df_all = pd.DataFrame(all_artists_data)

# Preview the data
print(f"\nSample data:")
df_all.head(10)


Sample data:


,year,artist
0,2022,12th Planet
1,2022,1991
2,2022,5GODZ
3,2022,A-Trak
4,2022,Craze
5,2022,A-Trak
6,2022,A.M.C
7,2022,AC Slater
8,2022,Chris Lorenzo
9,2022,ACIDTWIINS


In [35]:
# Save data to CSV files by year
for year in edc_web_urls.keys():
    df_year = df_all[df_all['year'] == year]
    filename = f'../data/extract/{year}_edc_artists.csv'
    df_year.to_csv(filename, index=False)
    print(f"Saved {len(df_year)} artists to {filename}")

Saved 1413 artists to ../data/extract/2022_edc_artists.csv
Saved 1110 artists to ../data/extract/2023_edc_artists.csv
Saved 1275 artists to ../data/extract/2024_edc_artists.csv
Saved 1287 artists to ../data/extract/2025_edc_artists.csv


# Collect 2026 Vegas EDM DJ Residencies and 2025's top 101 producers from:
https://electronic.vegas/las-vegas-resident-edm-dj-list/ and https://top101producers.com/2025
- Scraped using Thunderbit

# Top EDM Agencies and their list of artists
https://www.unitedtalent.com/talent/music UTA

https://www.insomniac.com/music/artists/ Insomniac - used Thunderbit(owns and operates EDC)

https://www.caa.com/entertainmenttalent/touring/search#electronic CAA

https://www.teamwass.com/artist-roster/genre/electronic Wasserman

# UTA

In [37]:
uta_web_urls = 'https://www.unitedtalent.com/talent/music'

page = requests.get(uta_web_urls)
soup = BeautifulSoup(page.content, 'html')

print(soup.prettify()[:1000])  # Print first 1000 characters of the UTA page for inspection

<!DOCTYPE html>
<html>
 <head>
  <meta charset="utf-8" data-next-head=""/>
  <meta content="width=device-width" data-next-head="" name="viewport"/>
  <link as="style" href="/_next/static/css/e7cc8a617bf2fc70.css" rel="preload"/>
  <link data-n-g="" href="/_next/static/css/e7cc8a617bf2fc70.css" rel="stylesheet"/>
  <link as="style" href="/_next/static/css/8aee48eb52f4c731.css" rel="preload"/>
  <link data-n-p="" href="/_next/static/css/8aee48eb52f4c731.css" rel="stylesheet"/>
  <link as="style" href="/_next/static/css/3371aa8f35d55aef.css" rel="preload"/>
  <link data-n-p="" href="/_next/static/css/3371aa8f35d55aef.css" rel="stylesheet"/>
  <noscript data-n-css="">
  </noscript>
  <script defer="" nomodule="" src="/_next/static/chunks/polyfills-42372ed130431b0a.js">
  </script>
  <script defer="" src="/_next/static/chunks/webpack-2f0997b4e014d2fc.js">
  </script>
  <script defer="" src="/_next/static/chunks/framework-f1c3457f730414be.js">
  </script>
  <script defer="" src="/_next/stati

In [38]:
# soup.find(class_='text-mobileParagraphLargeBold md:text-paragraphLargeBold py-2 md:py-1')
# example of a html line, we want the text "Trivecta" <div class="text-mobileParagraphLargeBold md:text-paragraphLargeBold py-2 md:py-1">Trivecta</div>
# headless = no popup window
options = webdriver.ChromeOptions()
options.add_argument("--headless")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

In [39]:
driver.get("https://www.unitedtalent.com/talent/music")
time.sleep(5)  # wait for JS to load

soup = BeautifulSoup(driver.page_source, "html.parser")

# grab artist name divs
artists = soup.find_all("div", class_="text-mobileParagraphLargeBold")

uta_artist_names = [a.text.strip() for a in artists if a.text.strip()]

driver.quit()

uta_artist_names[:10]

['[IVY]',
 '1010benja',
 '15  15',
 '2BYG',
 '2manydjs',
 '3 Doors Down',
 '33  Below',
 '347aidan',
 '49th  & Main',
 '802']

In [40]:
df_uta = pd.DataFrame(uta_artist_names, columns=['artist'])
df_uta['agency'] = 'uta'

print("\nSample data:")
df_uta.head(10)

uta_filename = f'../data/extract/uta_artists.csv'
df_uta.to_csv(uta_filename, index=False)
print(f"Saved {len(df_uta)} artists to {uta_filename}")


Sample data:
Saved 1835 artists to ../data/extract/uta_artists.csv


# CAA

In [68]:
options = webdriver.ChromeOptions()
options.add_argument("--headless")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

driver.get("https://www.caa.com/entertainmenttalent/touring/search#electronic")

# Scroll to load all artists
last_height = driver.execute_script("return document.body.scrollHeight")
while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)  # wait for JS to load more artists
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break  # no more new content
    last_height = new_height

# Parse page
soup = BeautifulSoup(driver.page_source, "html.parser")
artist_blocks = soup.find_all("div", class_="col-md-3 col-sm-4 col-xs-12 artist-block")
caa_artists = [b.find("span", class_="artist-name").text.strip() for b in artist_blocks if b.find("span", class_="artist-name")]

driver.quit()

# Save to csv
df_caa = pd.DataFrame(caa_artists, columns=['artist'])
df_caa['agency'] = 'caa'
df_caa.to_csv('../data/extract/caa_artists.csv', index=False)
print(f"Saved {len(df_caa)} artists")
print(caa_artists[:10])

Saved 228 artists
['Chris Stussy', 'Chromatics', 'Chromeo', 'Claxy', 'Cloonee', 'Club Heart Broken', 'Clüb De Combat', 'Collect 200', 'Colyn', 'Cristoph']
